In [9]:
import polars as pl
from pathlib import Path

DATA_DIR = Path("./csiro-biomass")

train = pl.read_csv(DATA_DIR / "train.csv")
test = pl.read_csv(DATA_DIR / "test.csv")
train

sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
str,str,str,str,str,f64,f64,str,f64
"""ID1011485656__Dry_Clover_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Clover_g""",0.0
"""ID1011485656__Dry_Dead_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Dead_g""",31.9984
"""ID1011485656__Dry_Green_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Green_g""",16.2751
"""ID1011485656__Dry_Total_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Total_g""",48.2735
"""ID1011485656__GDM_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""GDM_g""",16.275
…,…,…,…,…,…,…,…,…
"""ID983582017__Dry_Clover_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Clover_g""",0.0
"""ID983582017__Dry_Dead_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Dead_g""",0.0
"""ID983582017__Dry_Green_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Green_g""",40.94


In [10]:
train_cleaned = train.select("image_path", "target_name", "target").to_pandas()
test_cleaned = test.select("image_path", "target_name").with_columns(pl.lit(0).alias("target")).to_pandas()
train_cleaned.head(10)

,image_path,target_name,target
0,train/ID1011485656.jpg,Dry_Clover_g,0.0000
1,train/ID1011485656.jpg,Dry_Dead_g,31.9984
2,train/ID1011485656.jpg,Dry_Green_g,16.2751
3,train/ID1011485656.jpg,Dry_Total_g,48.2735
4,train/ID1011485656.jpg,GDM_g,16.2750
5,train/ID1012260530.jpg,Dry_Clover_g,0.0000
6,train/ID1012260530.jpg,Dry_Dead_g,0.0000
7,train/ID1012260530.jpg,Dry_Green_g,7.6000
8,train/ID1012260530.jpg,Dry_Total_g,7.6000
9,train/ID1012260530.jpg,GDM_g,7.6000


In [11]:
test_cleaned

,image_path,target_name,target
0,test/ID1001187975.jpg,Dry_Clover_g,0
1,test/ID1001187975.jpg,Dry_Dead_g,0
2,test/ID1001187975.jpg,Dry_Green_g,0
3,test/ID1001187975.jpg,Dry_Total_g,0
4,test/ID1001187975.jpg,GDM_g,0


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
train_df, val_df = train_test_split(train_cleaned, test_size=0.2, random_state=42)
train_df["target_name"] = label_encoder.fit_transform(train_df["target_name"])
val_df["target_name"] = label_encoder.transform(val_df["target_name"])
test_cleaned["target_name"] = label_encoder.transform(test_cleaned["target_name"])
train_df.head(10)

,image_path,target_name,target
1723,train/ID94564238.jpg,3,17.3000
175,train/ID1159071020.jpg,0,1.3191
886,train/ID1962197151.jpg,1,0.0000
1604,train/ID793526563.jpg,4,102.8303
481,train/ID147528735.jpg,1,3.0000
1083,train/ID227847873.jpg,3,40.1600
1747,train/ID968643034.jpg,2,9.0000
1473,train/ID679913293.jpg,3,21.2454
879,train/ID1953218650.jpg,4,10.5000
450,train/ID1463690813.jpg,0,0.0000


In [13]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class BiomassDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        image = Image.open(self.img_dir / img_path)
        target_name = row['target_name']
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['target'], dtype=torch.float32)
        return image, target_name, label

train_dataset = BiomassDataset(train_df, DATA_DIR)
val_dataset = BiomassDataset(val_df, DATA_DIR)
test_dataset = BiomassDataset(test_cleaned, DATA_DIR)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=16, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=16, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=16, pin_memory=True)

for images, target_names, labels in train_loader:
    print(images.shape, target_names.shape, labels.shape)
    break

torch.Size([32, 3, 224, 224]) torch.Size([32]) torch.Size([32])


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim

# Detect CUDA device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

class BiomassModel(nn.Module):
    def __init__(self, num_targets, num_target_names=5):
        super(BiomassModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # Embedding for target_name
        self.target_name_embedding = nn.Embedding(num_target_names, 16)
        
        self.classifier = nn.Sequential(
            nn.Linear(32 * 56 * 56 + 16, 128),
            nn.ReLU(),
            nn.Linear(128, num_targets),
        )

    def forward(self, x, target_names):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        # Embed target_names
        target_name_emb = self.target_name_embedding(target_names)
        # Concatenate image features with target_name embedding
        x = torch.cat([x, target_name_emb], dim=1)
        x = self.classifier(x)
        return x
    
num_targets = 1
model = BiomassModel(num_targets, num_target_names=5)
model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, target_names, labels in train_loader:
        images = images.to(device)
        target_names = target_names.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images, target_names)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    epoch_loss = running_loss / len(train_dataset)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')


Using device: cuda
Epoch 1/10, Loss: 719.1895
Epoch 2/10, Loss: 643.5688
Epoch 3/10, Loss: 601.0115
Epoch 4/10, Loss: 544.0843
Epoch 5/10, Loss: 478.1121
Epoch 6/10, Loss: 451.0427
Epoch 7/10, Loss: 333.6447
Epoch 8/10, Loss: 314.6251
Epoch 9/10, Loss: 271.0515
Epoch 10/10, Loss: 238.4127


In [17]:
outputs = []
model.eval()
with torch.no_grad():
    for images, target_names, _ in test_loader:
        images = images.to(device)
        target_names = target_names.to(device)
        preds = model(images, target_names)
        preds = preds.cpu()
        outputs.append(preds)

outputs = torch.cat(outputs, dim=0).numpy()
outputs

array([[ 6.2250347],
       [11.783444 ],
       [21.944275 ],
       [39.75187  ],
       [29.29157  ]], dtype=float32)

In [18]:
sample_submission = pl.read_csv(DATA_DIR / "sample_submission.csv")
# outputs is already a concatenated tensor from the previous cell
sample_submission = sample_submission.with_columns(pl.Series("target", outputs.flatten()))
sample_submission

sample_id,target
str,f32
"""ID1001187975__Dry_Clover_g""",6.225035
"""ID1001187975__Dry_Dead_g""",11.783444
"""ID1001187975__Dry_Green_g""",21.944275
"""ID1001187975__Dry_Total_g""",39.751869
"""ID1001187975__GDM_g""",29.291571
